In [38]:
# Cell 4 - FINAL: saves features + group labels + person labels + team_flags
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json

FEAT_DIR  = Path("/kaggle/input/datasets/bavleyhesham/my-processed-dataset/features")
ANNO_ROOT = Path("/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos")
SEQ_DIR   = Path("/kaggle/working/processed/sequences")
SEQ_DIR.mkdir(parents=True, exist_ok=True)

N_KEYFRAMES = 9
N_PLAYERS   = 12
N_SUBFRAMES = 20
MAX_PLAYERS = 12    # paper uses exactly 12, no need to pad to 13
FEAT_DIM    = 512
IMG_WIDTH   = 1280  # volleyball court frame width

GROUP_VOCAB = {
    "r_set": 0, "r_spike": 1, "r-pass": 2, "r_winpoint": 3,
    "l_set": 4, "l-spike": 5, "l_pass": 6, "l_winpoint": 7
}
ACTION_VOCAB = {
    "waiting": 0, "setting": 1, "digging": 2, "falling": 3,
    "spiking": 4, "blocking": 5, "jumping": 6,
    "moving":  7, "standing": 8, "unknown": 9
}

# ── Build annotation lookup ───────────────────────────────────────────────────
# Line format: "frame.jpg  group  x y w h action  x y w h action ..."
# We need: group_label, per-player action, per-player x-center (for team flag)
print("Building annotation lookup...")
anno_dict = {}

for vid_dir in sorted(ANNO_ROOT.iterdir()):
    if not vid_dir.is_dir(): continue
    anno_file = vid_dir / "annotations.txt"
    if not anno_file.exists(): continue
    with open(anno_file) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) < 7: continue

            frame_key   = parts[0]   # "10015.jpg"
            group_label = parts[1]   # "r_spike"

            actions   = []
            x_centers = []
            # Each player block: x y w h action  (5 fields)
            for i in range(2, len(parts), 5):
                if i + 4 >= len(parts): break
                x      = int(parts[i])
                w      = int(parts[i + 2])
                action = parts[i + 4]
                x_centers.append(x + w // 2)   # center x of bounding box
                actions.append(action)

            anno_dict[frame_key] = {
                "group":     group_label,
                "actions":   actions,
                "x_centers": x_centers   # used to assign team flags
            }

print(f"Annotation entries : {len(anno_dict)}")

# Quick sanity check
sample = list(anno_dict.values())[0]
print(f"Sample — group:{sample['group']}  "
      f"n_players:{len(sample['actions'])}  "
      f"x_centers:{sample['x_centers']}")

# ── Feature file → annotation key match test ─────────────────────────────────
feat_files = sorted(FEAT_DIR.glob("*.npy"))
hits = sum(1 for f in feat_files[:50]
           if f.stem.replace("_features","") + ".jpg" in anno_dict)
print(f"Key match (first 50): {hits}/50")
if hits == 0:
    raise RuntimeError("No keys matched — check paths.")

# ── Reshape: (2160, 512) → (9, 12, 512) ──────────────────────────────────────
def reshape_features(raw):
    raw = raw.astype(np.float32)
    # (9 keyframes × 12 players × 20 subframes, 512)
    raw = raw.reshape(N_KEYFRAMES, N_PLAYERS, N_SUBFRAMES, FEAT_DIM)
    return raw.mean(axis=2)   # (9, 12, 512) — average over 20 sub-frames

# ── Allocate memmaps ──────────────────────────────────────────────────────────
total     = len(feat_files)
feat_mm   = np.lib.format.open_memmap(
    SEQ_DIR/"train_features.npy", mode="w+",
    dtype="float32", shape=(total, N_KEYFRAMES, MAX_PLAYERS, FEAT_DIM))
plabel_mm = np.lib.format.open_memmap(
    SEQ_DIR/"train_person_labels.npy", mode="w+",
    dtype="int16",   shape=(total, MAX_PLAYERS))
glabel_mm = np.lib.format.open_memmap(
    SEQ_DIR/"train_group_labels.npy", mode="w+",
    dtype="int16",   shape=(total,))
# NEW: team_flags — 1=left team (x<640), 0=right team (x>=640)
flags_mm  = np.lib.format.open_memmap(
    SEQ_DIR/"train_team_flags.npy", mode="w+",
    dtype="int16",   shape=(total, MAX_PLAYERS))

print(f"Memmap: {feat_mm.nbytes/1e9:.2f} GB on disk")

# ── Main loop ─────────────────────────────────────────────────────────────────
written = skipped = unknown_group = 0

for npy_file in tqdm(feat_files):
    try:
        raw = np.load(npy_file, allow_pickle=False)
    except Exception as e:
        skipped += 1
        continue

    if raw.shape[0] != N_KEYFRAMES * N_PLAYERS * N_SUBFRAMES:
        skipped += 1
        continue

    features  = reshape_features(raw)          # (9, 12, 512)
    anno_key  = npy_file.stem.replace("_features","") + ".jpg"
    entry     = anno_dict.get(anno_key, None)

    if entry is None:
        group_idx     = -1
        person_labels = [ACTION_VOCAB["unknown"]] * MAX_PLAYERS
        team_flags    = [0] * MAX_PLAYERS   # default all team B
        unknown_group += 1
    else:
        group_idx = GROUP_VOCAB.get(entry["group"], -1)
        if group_idx == -1:
            unknown_group += 1

        # Person action labels
        raw_acts      = entry["actions"][:MAX_PLAYERS]
        person_labels = [ACTION_VOCAB.get(a, ACTION_VOCAB["unknown"])
                         for a in raw_acts]
        while len(person_labels) < MAX_PLAYERS:
            person_labels.append(ACTION_VOCAB["unknown"])

        # Team flags from x-center: left half=1 (TeamA), right half=0 (TeamB)
        raw_x      = entry["x_centers"][:MAX_PLAYERS]
        team_flags = [1 if x < (IMG_WIDTH // 2) else 0 for x in raw_x]
        while len(team_flags) < MAX_PLAYERS:
            team_flags.append(0)

    feat_mm  [written] = features
    plabel_mm[written] = person_labels
    glabel_mm[written] = group_idx
    flags_mm [written] = team_flags
    written += 1
    del features

del feat_mm, plabel_mm, glabel_mm, flags_mm

with open(SEQ_DIR/"group_vocab.json",  "w") as f: json.dump(GROUP_VOCAB,  f)
with open(SEQ_DIR/"action_vocab.json", "w") as f: json.dump(ACTION_VOCAB, f)

# ── Report ────────────────────────────────────────────────────────────────────
g = np.load(SEQ_DIR/"train_group_labels.npy")
fl = np.load(SEQ_DIR/"train_team_flags.npy")
inv = {v: k for k, v in GROUP_VOCAB.items()}
unique, counts = np.unique(g, return_counts=True)

print(f"\n✅ Cell 4 done!")
print(f"   Written        : {written}")
print(f"   Skipped        : {skipped}")
print(f"   Valid for train: {written - unknown_group}")
print(f"\n   Group label distribution:")
for v, c in zip(unique, counts):
    print(f"     {inv.get(int(v),'unknown/-1'):12s}: {c}")

print(f"\n   Team flag sample (first sample):")
print(f"     flags = {fl[0].tolist()}")
print(f"     TeamA (x<640): {fl[0].sum()} players")
print(f"     TeamB (x>=640): {(fl[0]==0).sum()} players")
print(f"\n   Feature shape: {np.load(SEQ_DIR/'train_features.npy', mmap_mode='r').shape}")

Building annotation lookup...
Annotation entries : 4131
Sample — group:r_winpoint  n_players:12  x_centers:[295, 319, 375, 452, 558, 552, 754, 738, 802, 754, 871, 920]
Key match (first 50): 50/50
Memmap: 0.86 GB on disk


100%|██████████| 3880/3880 [00:24<00:00, 158.58it/s]


✅ Cell 4 done!
   Written        : 3677
   Skipped        : 203
   Valid for train: 3034

   Group label distribution:
     unknown/-1  : 643
     r_set       : 716
     r_spike     : 493
     r-pass      : 600
     r_winpoint  : 194
     l_set       : 492
     l-spike     : 493
     l_winpoint  : 249

   Team flag sample (first sample):
     flags = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
     TeamA (x<640): 6 players
     TeamB (x>=640): 6 players

   Feature shape: (3880, 9, 12, 512)


In [41]:
# Cell 5 - FIXED: float32 forced in GroupLSTM, exact paper architecture
import torch
import torch.nn as nn
import json
from pathlib import Path

SEQ_DIR = Path("/kaggle/working/processed/sequences")
with open(SEQ_DIR/"group_vocab.json")  as f: GROUP_VOCAB  = json.load(f)
with open(SEQ_DIR/"action_vocab.json") as f: ACTION_VOCAB = json.load(f)

NUM_GROUP_CLASSES  = len(GROUP_VOCAB)
NUM_ACTION_CLASSES = len(ACTION_VOCAB)


class PersonLSTM(nn.Module):
    def __init__(self, feat_dim=512, hidden=3000,
                 num_action_classes=10, num_layers=2):
        super().__init__()
        self.hidden = hidden
        self.lstm   = nn.LSTM(feat_dim, hidden, num_layers=num_layers,
                              batch_first=True, dropout=0.5)
        self.norm        = nn.LayerNorm(hidden)
        self.action_head = nn.Linear(hidden, num_action_classes)
        self._init()

    def _init(self):
        for name, p in self.lstm.named_parameters():
            if   "weight_ih" in name: nn.init.xavier_uniform_(p)
            elif "weight_hh" in name: nn.init.orthogonal_(p)
            elif "bias"      in name: nn.init.zeros_(p)
        nn.init.xavier_uniform_(self.action_head.weight)
        nn.init.zeros_(self.action_head.bias)

    def forward(self, x):
        # x: (B, T, 512)
        h, _  = self.lstm(x)        # (B, T, 3000)
        h     = self.norm(h)        # stabilise magnitude
        return h, self.action_head(h)


class TwoTeamGroupLSTM(nn.Module):
    def __init__(self, person_feat=512, person_hidden=3000,
                 fc_dim=3000, group_hidden=500,
                 num_group_classes=8, num_layers=2):
        super().__init__()
        concat_dim   = person_feat + person_hidden   # 3512
        two_team_dim = concat_dim * 2                # 7024

        self.pool_norm = nn.LayerNorm(two_team_dim)
        self.fc        = nn.Sequential(
            nn.Linear(two_team_dim, fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )
        self.lstm = nn.LSTM(fc_dim, group_hidden, num_layers=num_layers,
                            batch_first=True, dropout=0.3)
        self.head = nn.Linear(group_hidden, num_group_classes)
        self._init()

    def _init(self):
        nn.init.xavier_uniform_(self.fc[0].weight)
        nn.init.zeros_(self.fc[0].bias)
        for name, p in self.lstm.named_parameters():
            if   "weight_ih" in name: nn.init.xavier_uniform_(p)
            elif "weight_hh" in name: nn.init.orthogonal_(p)
            elif "bias"      in name: nn.init.zeros_(p)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, cnn_feats, h_seq, team_flags):
        # ── CRITICAL: run entirely in float32 — autocast fp16 causes NaN ──────
        # Random LSTM init + fp16 = overflow. float32 is non-negotiable here.
        with torch.amp.autocast("cuda", enabled=False):
            cnn_feats = cnn_feats.float()    # (B, T, K, 512)
            h_seq     = h_seq.float()        # (B, T, K, 3000)

            # eq(7): P_tk = concat(CNN, h) per player per timestep
            P     = torch.cat([cnn_feats, h_seq], dim=-1)  # (B, T, K, 3512)

            # flags: (B, K) → (B, 1, K, 1)
            flags = team_flags.float().unsqueeze(1).unsqueeze(-1)

            # ── Max-pool per team (paper eq 8, pure float32) ──────────────────
            # Safe NEG for float32 — won't overflow
            NEG   = -1e6

            # Team A (flag=1)
            teamA = P * flags      + (1.0 - flags) * NEG
            teamA = teamA.max(dim=2)[0]                    # (B, T, 3512)

            # Team B (flag=0)
            teamB = P * (1-flags)  + flags * NEG
            teamB = teamB.max(dim=2)[0]                    # (B, T, 3512)

            # Replace any -inf/nan left over (empty team edge case)
            teamA = torch.nan_to_num(teamA, nan=0.0, neginf=0.0)
            teamB = torch.nan_to_num(teamB, nan=0.0, neginf=0.0)

            # Concat → LayerNorm → FC → LSTM → head
            Z     = torch.cat([teamA, teamB], dim=-1)      # (B, T, 7024)
            Z     = self.pool_norm(Z)
            Z     = self.fc(Z)                             # (B, T, 3000)
            g, _  = self.lstm(Z)                           # (B, T, 500)
            return self.head(g[:, -1, :])                  # (B, classes)


class HierarchicalModel(nn.Module):
    def __init__(self, feat_dim=512, person_hidden=3000,
                 fc_dim=3000, group_hidden=500,
                 num_action_classes=10, num_group_classes=8):
        super().__init__()
        self.feat_dim      = feat_dim
        self.person_hidden = person_hidden
        self.person_lstm   = PersonLSTM(feat_dim, person_hidden, num_action_classes)
        self.group_lstm    = TwoTeamGroupLSTM(feat_dim, person_hidden,
                                              fc_dim, group_hidden, num_group_classes)

    def forward(self, features, team_flags, stage=2):
        B, T, K, F = features.shape
        x          = features.permute(0,2,1,3).reshape(B*K, T, F)
        h, a       = self.person_lstm(x)
        h_seq      = h.reshape(B,K,T,self.person_hidden).permute(0,2,1,3)
        a_logits   = a.reshape(B,K,T,-1).permute(0,2,1,3)
        if stage == 1:
            return a_logits
        return self.group_lstm(features, h_seq, team_flags), a_logits


# ── Smoke-test: verify zero NaN before training ────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = HierarchicalModel(
    num_action_classes=NUM_ACTION_CLASSES,
    num_group_classes=NUM_GROUP_CLASSES).to(device)

dummy  = torch.randn(4, 9, 12, 512, device=device)
flags  = torch.zeros(4, 12, dtype=torch.long, device=device)
flags[:, :6] = 1   # left team

# Test under autocast exactly as training will use it
with torch.amp.autocast("cuda"):
    g_out, a_out = model(dummy, flags, stage=2)

print("NaN check (must be False):")
print(f"  group logits : {torch.isnan(g_out).any().item()}")
print(f"  action logits: {torch.isnan(a_out).any().item()}")
assert not torch.isnan(g_out).any(), "Still NaN — something is wrong"
print(f"\n✅ Cell 5 passed!")
print(f"   group  : {g_out.shape}")
print(f"   actions: {a_out.shape}")
print(f"   Params : {sum(p.numel() for p in model.parameters()):,}")

NaN check (must be False):
  group logits : False
  action logits: False

✅ Cell 5 passed!
   group  : torch.Size([4, 8])
   actions: torch.Size([4, 9, 12, 10])
   Params : 144,329,066


In [42]:
# Cell 6 - FIXED: weighted person loss + float32 Stage 2
import numpy as np
import torch, time, json
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

CFG = dict(
    lr              = 1e-4,
    weight_decay    = 1e-4,
    stage1_epochs   = 30,
    stage2_epochs   = 50,
    batch_size      = 32,
    val_split       = 0.2,
    num_workers     = 2,
    standing_keep   = 0.15,
    standing_thresh = 0.5,
    ckpt_dir = Path("/kaggle/working/processed/checkpoints"),
    seq_dir  = Path("/kaggle/working/processed/sequences"),
)
CFG["ckpt_dir"].mkdir(exist_ok=True)

with open(CFG["seq_dir"]/"group_vocab.json")  as f: GROUP_VOCAB  = json.load(f)
with open(CFG["seq_dir"]/"action_vocab.json") as f: ACTION_VOCAB = json.load(f)
NUM_GROUP_CLASSES  = len(GROUP_VOCAB)
NUM_ACTION_CLASSES = len(ACTION_VOCAB)
UNKNOWN_IDX        = ACTION_VOCAB["unknown"]
STANDING_IDX       = ACTION_VOCAB["standing"]
INV_VOCAB          = {v: k for k, v in GROUP_VOCAB.items()}


class VolleyballDataset(Dataset):
    def __init__(self, seq_dir, standing_keep=0.15, standing_thresh=0.5):
        seq_dir       = Path(seq_dir)
        self.features = np.load(seq_dir/"train_features.npy",      mmap_mode="r")
        self.p_labels = np.load(seq_dir/"train_person_labels.npy", mmap_mode="r")
        self.g_labels = np.load(seq_dir/"train_group_labels.npy",  mmap_mode="r")
        self.t_flags  = np.load(seq_dir/"train_team_flags.npy",    mmap_mode="r")

        valid         = np.where(self.g_labels != -1)[0]
        p_valid       = self.p_labels[valid]
        stand_frac    = (p_valid == STANDING_IDX).mean(axis=1)
        is_heavy      = stand_frac > standing_thresh
        rng           = np.random.default_rng(42)
        n_keep        = max(1, int(is_heavy.sum() * standing_keep))
        kept          = rng.choice(valid[is_heavy], size=n_keep, replace=False)
        self.indices  = np.sort(np.concatenate([valid[~is_heavy], kept]))

        print(f"  Valid            : {len(valid)}")
        print(f"  Standing kept    : {n_keep}/{is_heavy.sum()}")
        print(f"  Final dataset    : {len(self.indices)}")

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return (
            torch.from_numpy(np.array(self.features[idx])).float(),
            torch.from_numpy(np.array(self.p_labels[idx])).long(),
            torch.tensor(int(self.g_labels[idx]), dtype=torch.long),
            torch.from_numpy(np.array(self.t_flags[idx])).long(),
        )


print("Loading dataset...")
full_ds = VolleyballDataset(CFG["seq_dir"], CFG["standing_keep"], CFG["standing_thresh"])
n_val   = max(1, int(len(full_ds) * CFG["val_split"]))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=CFG["num_workers"],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"]*2,
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)
print(f"  Train:{n_train}  Val:{n_val}  Batches:{len(train_loader)}/{len(val_loader)}")

# ── Compute class weights from training data to fix standing dominance ─────────
print("\nComputing person action class weights...")
p_all   = full_ds.p_labels[full_ds.indices]      # (N, 12)
p_flat  = p_all.flatten()
p_valid = p_flat[p_flat != UNKNOWN_IDX]

counts  = np.bincount(p_valid, minlength=NUM_ACTION_CLASSES).astype(np.float32)
counts[counts == 0] = 1   # avoid div/0
# Inverse frequency — down-weights standing, up-weights rare actions
person_weights = (p_valid.shape[0] / (NUM_ACTION_CLASSES * counts))
person_weights[UNKNOWN_IDX] = 0.0   # unknown gets zero weight anyway

print("  Person action weights:")
inv_act = {v: k for k, v in ACTION_VOCAB.items()}
for i, w in enumerate(person_weights):
    print(f"    {inv_act[i]:10s}: {w:.3f}  (n={int(counts[i])})")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = HierarchicalModel(
    num_action_classes=NUM_ACTION_CLASSES,
    num_group_classes=NUM_GROUP_CLASSES).to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"\nDataParallel: {torch.cuda.device_count()} GPUs")

# Weighted person loss — fixes standing dominance
person_criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(person_weights, dtype=torch.float32).to(device),
    ignore_index=UNKNOWN_IDX)
group_criterion  = nn.CrossEntropyLoss()

def get_core(m): return m.module if isinstance(m, nn.DataParallel) else m
def freeze(mod): [p.requires_grad_(False) for p in mod.parameters()]


def run_epoch(loader, optimizer, scaler, stage, train=True):
    model.train() if train else model.eval()
    total_loss = total_correct = total_samples = nan_count = 0
    t0 = time.time()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for feats, p_labels, g_labels, t_flags in loader:
            feats    = feats.to(device,    non_blocking=True)
            p_labels = p_labels.to(device, non_blocking=True)
            g_labels = g_labels.to(device, non_blocking=True)
            t_flags  = t_flags.to(device,  non_blocking=True)

            # Stage 1: autocast is fine (PersonLSTM only)
            # Stage 2: GroupLSTM forces float32 internally via autocast(enabled=False)
            # So we can safely use autocast for the whole forward in both stages
            with torch.amp.autocast("cuda"):
                if stage == 1:
                    a_logits = model(feats, t_flags, stage=1)
                    T        = a_logits.shape[1]
                    p_tgt    = p_labels.unsqueeze(1).expand(-1, T, -1)
                    loss     = person_criterion(
                        a_logits.reshape(-1, NUM_ACTION_CLASSES),
                        p_tgt.reshape(-1))
                    mask     = p_tgt.reshape(-1) != UNKNOWN_IDX
                    correct  = (a_logits.reshape(-1, NUM_ACTION_CLASSES)
                                .argmax(-1)[mask] ==
                                p_tgt.reshape(-1)[mask]).sum().item() \
                               if mask.any() else 0
                    samples  = mask.sum().item() if mask.any() else 1

                else:
                    g_logits, a_logits = model(feats, t_flags, stage=2)
                    T        = a_logits.shape[1]
                    p_tgt    = p_labels.unsqueeze(1).expand(-1, T, -1)
                    g_loss   = group_criterion(g_logits, g_labels)
                    aux_loss = person_criterion(
                        a_logits.reshape(-1, NUM_ACTION_CLASSES),
                        p_tgt.reshape(-1))
                    loss     = g_loss + 0.5 * aux_loss
                    correct  = (g_logits.argmax(-1) == g_labels).sum().item()
                    samples  = g_labels.size(0)

            if torch.isnan(loss):
                nan_count += 1
                continue

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss    += loss.item() * feats.size(0)
            total_correct += correct
            total_samples += samples

    if nan_count: print(f"    ⚠ {nan_count} NaN batches")
    n = len(loader.dataset)
    return (total_loss / max(n, 1),
            100.0 * total_correct / max(total_samples, 1),
            time.time() - t0)


# ── Stage 1 ───────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("STAGE 1: PersonLSTM  (weighted loss)")
print("="*55)
opt1   = torch.optim.Adam(get_core(model).person_lstm.parameters(),
                          lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched1 = torch.optim.lr_scheduler.StepLR(opt1, step_size=15, gamma=0.1)
scaler = torch.amp.GradScaler("cuda")
best1  = 0.

for ep in range(1, CFG["stage1_epochs"]+1):
    tr = run_epoch(train_loader, opt1, scaler, 1, True)
    vl = run_epoch(val_loader,   opt1, scaler, 1, False)
    sched1.step()
    tag = " ◄" if vl[1] > best1 else ""
    print(f"[S1 {ep:02d}] loss={tr[0]:.4f} acc={tr[1]:.1f}% | "
          f"val={vl[0]:.4f} acc={vl[1]:.1f}% ({tr[2]:.0f}s){tag}")
    if vl[1] > best1:
        best1 = vl[1]
        torch.save(get_core(model).person_lstm.state_dict(),
                   CFG["ckpt_dir"]/"best_stage1.pt")
print(f"✅ Stage 1 best: {best1:.1f}%")

# ── Stage 2 ───────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("STAGE 2: GroupLSTM  (PersonLSTM frozen)")
print("="*55)
get_core(model).person_lstm.load_state_dict(
    torch.load(CFG["ckpt_dir"]/"best_stage1.pt", map_location=device))
freeze(get_core(model).person_lstm)
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

opt2   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched2 = torch.optim.lr_scheduler.StepLR(opt2, step_size=20, gamma=0.1)
scaler = torch.amp.GradScaler("cuda")
best2  = 0.

for ep in range(1, CFG["stage2_epochs"]+1):
    tr = run_epoch(train_loader, opt2, scaler, 2, True)
    vl = run_epoch(val_loader,   opt2, scaler, 2, False)
    sched2.step()
    tag = " ◄" if vl[1] > best2 else ""
    print(f"[S2 {ep:02d}] loss={tr[0]:.4f} acc={tr[1]:.1f}% | "
          f"val={vl[0]:.4f} acc={vl[1]:.1f}% ({tr[2]+vl[2]:.0f}s){tag}")
    if vl[1] > best2:
        best2 = vl[1]
        torch.save({"epoch": ep, "val_acc": vl[1],
                    "model_state": get_core(model).state_dict(),
                    "group_vocab": GROUP_VOCAB, "action_vocab": ACTION_VOCAB},
                   CFG["ckpt_dir"]/"best_full_model.pt")

print(f"\n✅ Done!  Best val acc: {best2:.1f}%  (paper: 51.1%)")

# ── Per-class breakdown ───────────────────────────────────────────────────────
ckpt = torch.load(CFG["ckpt_dir"]/"best_full_model.pt", map_location=device)
get_core(model).load_state_dict(ckpt["model_state"])
model.eval()
preds, labels = [], []
with torch.no_grad():
    for feats, _, g_labels, t_flags in val_loader:
        g_out, _ = model(feats.to(device), t_flags.to(device), stage=2)
        preds.append(g_out.argmax(-1).cpu())
        labels.append(g_labels)
preds  = torch.cat(preds)
labels = torch.cat(labels)

print(f"\n{'Class':12s}  {'Acc':>7}  {'N':>5}")
print("-"*28)
for i in range(NUM_GROUP_CLASSES):
    m = labels == i
    if not m.any(): continue
    c = (preds[m] == i).sum().item()
    print(f"{INV_VOCAB[i]:12s}  {100*c/m.sum():>6.1f}%  {m.sum():>5}")
print(f"\nOverall: {100*(preds==labels).float().mean():.1f}%")

Loading dataset...
  Valid            : 3237
  Standing kept    : 313/2091
  Final dataset    : 1459
  Train:1168  Val:291  Batches:36/5

Computing person action class weights...
  Person action weights:
    waiting   : 0.392  (n=4344)
    setting   : 3.760  (n=453)
    digging   : 2.712  (n=628)
    falling   : 7.604  (n=224)
    spiking   : 2.872  (n=593)
    blocking  : 1.480  (n=1151)
    jumping   : 11.206  (n=152)
    moving    : 0.853  (n=1997)
    standing  : 0.227  (n=7491)
    unknown   : 0.000  (n=1)

DataParallel: 2 GPUs

STAGE 1: PersonLSTM  (weighted loss)
[S1 01] loss=3.5722 acc=22.8% | val=2.3735 acc=34.3% (10s) ◄
[S1 02] loss=2.1717 acc=21.7% | val=2.1767 acc=28.2% (10s)
[S1 03] loss=2.0751 acc=23.0% | val=2.0944 acc=17.2% (10s)
[S1 04] loss=2.0820 acc=24.6% | val=2.0391 acc=48.7% (10s) ◄
[S1 05] loss=2.0380 acc=27.0% | val=2.0620 acc=20.5% (9s)
[S1 06] loss=1.9792 acc=23.0% | val=2.0052 acc=32.1% (9s)
[S1 07] loss=1.9564 acc=22.9% | val=1.9825 acc=20.7% (9s)
[S1 08] l

In [44]:
# Model cell: compact, robust hierarchical model for person -> team -> group
import torch
import torch.nn as nn
import torch.nn.functional as F

# Device-aware autocast helper
def device_type():
    return "cuda" if torch.cuda.is_available() else "cpu"

class PersonLSTM(nn.Module):
    """
    Per-person temporal encoder.
    - feat_dim: input per-frame feature dim (e.g., 512)
    - hidden: LSTM hidden dim (kept moderate by default)
    - proj_dim: optional projection of final per-frame hidden to compact embedding
    - num_action_classes: action classification head per person per timestep
    """
    def __init__(self, feat_dim=512, hidden=512, proj_dim=256,
                 num_action_classes=10, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden = hidden
        self.proj_dim = proj_dim
        self.lstm = nn.LSTM(feat_dim, hidden, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden)
        self.action_head = nn.Linear(hidden, num_action_classes)
        # compact projection used by group model to reduce downstream cost
        self.project = nn.Linear(hidden, proj_dim) if proj_dim is not None else None
        self._init_weights()

    def _init_weights(self):
        for name, p in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(p)
            elif "weight_hh" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name:
                nn.init.zeros_(p)
        nn.init.xavier_uniform_(self.action_head.weight)
        nn.init.zeros_(self.action_head.bias)
        if self.project is not None:
            nn.init.xavier_uniform_(self.project.weight)
            nn.init.zeros_(self.project.bias)

    def forward(self, x):
        # x: (B*K, T, F)
        # Ensure LSTM runs in float32 for stability
        x = x.float()
        h, _ = self.lstm(x)            # (B*K, T, hidden)
        h = self.norm(h)
        a_logits = self.action_head(h) # (B*K, T, num_action_classes)
        if self.project is not None:
            p = self.project(h)        # (B*K, T, proj_dim)
        else:
            p = h
        return h, p, a_logits         # raw hidden, projected seq, action logits


class TemporalAttentionPool(nn.Module):
    """
    Simple temporal attention pooling across time dimension.
    Input: (B, T, D) -> Output: (B, D)
    """
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.q = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x, mask=None):
        # x: (B, T, D)
        scores = self.q(x).squeeze(-1)      # (B, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)  # (B, T)
        out = (attn.unsqueeze(-1) * x).sum(dim=1)  # (B, D)
        return out


class TwoTeamGroupLSTM(nn.Module):
    """
    Group-level model that pools per-person embeddings into two team embeddings,
    concatenates them, and runs a group-level LSTM (or optional attention).
    - proj_dim: dimension of per-person projected embedding (from PersonLSTM)
    - fc_dim: intermediate FC dim before group LSTM
    - group_hidden: hidden size of group LSTM
    - use_temporal_attn: whether to use temporal attention to aggregate group LSTM outputs
    """
    def __init__(self, proj_dim=256, fc_dim=512, group_hidden=256,
                 num_group_classes=8, num_layers=1, dropout=0.3,
                 use_temporal_attn=True):
        super().__init__()
        self.proj_dim = proj_dim
        two_team_dim = proj_dim * 2
        self.pool_norm = nn.LayerNorm(two_team_dim)
        self.fc = nn.Sequential(
            nn.Linear(two_team_dim, fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(fc_dim, group_hidden, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.use_temporal_attn = use_temporal_attn
        if use_temporal_attn:
            self.temporal_pool = TemporalAttentionPool(group_hidden, hidden=128)
            self.head = nn.Linear(group_hidden, num_group_classes)
        else:
            self.head = nn.Linear(group_hidden, num_group_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc[0].weight)
        nn.init.zeros_(self.fc[0].bias)
        for name, p in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(p)
            elif "weight_hh" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name:
                nn.init.zeros_(p)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, person_proj, team_flags):
        """
        person_proj: (B, T, K, proj_dim)  -- per-person projected sequence
        team_flags:  (B, K) with values 0/1 indicating team membership
        Returns: group logits (B, num_classes), optionally intermediate group sequence
        """
        # Force float32 for pooling and LSTM to avoid fp16 overflow
        with torch.amp.autocast(device_type(), enabled=False):
            person_proj = person_proj.float()   # (B, T, K, P)
            flags = team_flags.float().unsqueeze(1).unsqueeze(-1)  # (B,1,K,1)

            NEG = -1e6
            # Team A (flag=1)
            teamA = person_proj * flags + (1.0 - flags) * NEG
            teamA = teamA.max(dim=2)[0]  # (B, T, P)
            # Team B (flag=0)
            teamB = person_proj * (1.0 - flags) + flags * NEG
            teamB = teamB.max(dim=2)[0]  # (B, T, P)

            teamA = torch.nan_to_num(teamA, nan=0.0, neginf=0.0)
            teamB = torch.nan_to_num(teamB, nan=0.0, neginf=0.0)

            Z = torch.cat([teamA, teamB], dim=-1)  # (B, T, 2P)
            Z = self.pool_norm(Z)
            Z = self.fc(Z)                          # (B, T, fc_dim)
            g_seq, _ = self.lstm(Z)                 # (B, T, group_hidden)

            if self.use_temporal_attn:
                g_vec = self.temporal_pool(g_seq)   # (B, group_hidden)
            else:
                g_vec = g_seq[:, -1, :]             # last timestep

            logits = self.head(g_vec)               # (B, num_group_classes)
            return logits, g_seq


class HierarchicalModel(nn.Module):
    """
    Full model wrapper:
    - PersonLSTM encodes each person sequence (B*K, T, F) -> returns per-person proj (B*K, T, P)
    - HierarchicalModel.forward accepts features shaped (B, T, K, F)
    - Returns: group logits and action logits shaped (B, T, K, num_action_classes)
    """
    def __init__(self, feat_dim=512, person_hidden=512, proj_dim=256,
                 fc_dim=512, group_hidden=256,
                 num_action_classes=10, num_group_classes=8,
                 person_layers=2, group_layers=1, use_temporal_attn=True):
        super().__init__()
        self.feat_dim = feat_dim
        self.person_hidden = person_hidden
        self.person = PersonLSTM(feat_dim, person_hidden, proj_dim,
                                 num_action_classes, num_layers=person_layers)
        self.group = TwoTeamGroupLSTM(proj_dim, fc_dim, group_hidden,
                                      num_group_classes, num_layers=group_layers,
                                      use_temporal_attn=use_temporal_attn)

    def forward(self, features, team_flags, stage=2):
        """
        features: (B, T, K, F)
        team_flags: (B, K)
        stage:
          - 1: return action logits only (B, T, K, num_action_classes)
          - 2: return (group_logits, action_logits)
        """
        B, T, K, F = features.shape
        # reshape to per-person batch: (B*K, T, F)
        x = features.permute(0, 2, 1, 3).reshape(B * K, T, F)
        # Run person encoder in float32 to avoid fp16 LSTM issues
        with torch.amp.autocast(device_type(), enabled=False):
            h_raw, p_proj, a_logits = self.person(x)  # h_raw: (B*K, T, H), p_proj: (B*K, T, P)
        # reshape back: (B, T, K, P) and (B, T, K, num_action_classes)
        p_proj = p_proj.reshape(B, K, T, -1).permute(0, 2, 1, 3)
        a_logits = a_logits.reshape(B, K, T, -1).permute(0, 2, 1, 3)
        if stage == 1:
            return a_logits
        # group forward expects (B, T, K, P)
        g_logits, g_seq = self.group(p_proj, team_flags)
        return g_logits, a_logits


In [45]:
# Training cell (adapted to the refactored model)
import numpy as np
import torch, time, json
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

# --- config (same as before, adjust as needed) ---
CFG = dict(
    lr              = 1e-4,
    weight_decay    = 1e-4,
    stage1_epochs   = 30,
    stage2_epochs   = 50,
    batch_size      = 32,
    val_split       = 0.2,
    num_workers     = 2,
    standing_keep   = 0.15,
    standing_thresh = 0.5,
    ckpt_dir = Path("/kaggle/working/processed/checkpoints"),
    seq_dir  = Path("/kaggle/working/processed/sequences"),
)
CFG["ckpt_dir"].mkdir(exist_ok=True)

with open(CFG["seq_dir"]/"group_vocab.json")  as f: GROUP_VOCAB  = json.load(f)
with open(CFG["seq_dir"]/"action_vocab.json") as f: ACTION_VOCAB = json.load(f)
NUM_GROUP_CLASSES  = len(GROUP_VOCAB)
NUM_ACTION_CLASSES = len(ACTION_VOCAB)
UNKNOWN_IDX        = ACTION_VOCAB["unknown"]
STANDING_IDX       = ACTION_VOCAB["standing"]
INV_VOCAB          = {v: k for k, v in GROUP_VOCAB.items()}

# --- dataset class (unchanged) ---
class VolleyballDataset(Dataset):
    def __init__(self, seq_dir, standing_keep=0.15, standing_thresh=0.5):
        seq_dir       = Path(seq_dir)
        self.features = np.load(seq_dir/"train_features.npy",      mmap_mode="r")
        self.p_labels = np.load(seq_dir/"train_person_labels.npy", mmap_mode="r")
        self.g_labels = np.load(seq_dir/"train_group_labels.npy",  mmap_mode="r")
        self.t_flags  = np.load(seq_dir/"train_team_flags.npy",    mmap_mode="r")

        valid         = np.where(self.g_labels != -1)[0]
        p_valid       = self.p_labels[valid]
        stand_frac    = (p_valid == STANDING_IDX).mean(axis=1)
        is_heavy      = stand_frac > standing_thresh
        rng           = np.random.default_rng(42)
        n_keep        = max(1, int(is_heavy.sum() * standing_keep))
        kept          = rng.choice(valid[is_heavy], size=n_keep, replace=False)
        self.indices  = np.sort(np.concatenate([valid[~is_heavy], kept]))

        print(f"  Valid            : {len(valid)}")
        print(f"  Standing kept    : {n_keep}/{is_heavy.sum()}")
        print(f"  Final dataset    : {len(self.indices)}")

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return (
            torch.from_numpy(np.array(self.features[idx])).float(),
            torch.from_numpy(np.array(self.p_labels[idx])).long(),
            torch.tensor(int(self.g_labels[idx]), dtype=torch.long),
            torch.from_numpy(np.array(self.t_flags[idx])).long(),
        )

print("Loading dataset...")
full_ds = VolleyballDataset(CFG["seq_dir"], CFG["standing_keep"], CFG["standing_thresh"])
n_val   = max(1, int(len(full_ds) * CFG["val_split"]))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=CFG["num_workers"],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"]*2,
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)
print(f"  Train:{n_train}  Val:{n_val}  Batches:{len(train_loader)}/{len(val_loader)}")

# compute person class weights (same as your script)
p_all   = full_ds.p_labels[full_ds.indices]      # (N, 12)
p_flat  = p_all.flatten()
p_valid = p_flat[p_flat != UNKNOWN_IDX]
counts  = np.bincount(p_valid, minlength=NUM_ACTION_CLASSES).astype(np.float32)
counts[counts == 0] = 1
person_weights = (p_valid.shape[0] / (NUM_ACTION_CLASSES * counts))
person_weights[UNKNOWN_IDX] = 0.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# instantiate the refactored model with smaller defaults for stability
model = HierarchicalModel(
    feat_dim=512,
    person_hidden=512,
    proj_dim=256,
    fc_dim=512,
    group_hidden=256,
    num_action_classes=NUM_ACTION_CLASSES,
    num_group_classes=NUM_GROUP_CLASSES,
).to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"\nDataParallel: {torch.cuda.device_count()} GPUs")

person_criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(person_weights, dtype=torch.float32).to(device),
    ignore_index=UNKNOWN_IDX)
group_criterion  = nn.CrossEntropyLoss()

def get_core(m): return m.module if isinstance(m, nn.DataParallel) else m
def freeze(mod): [p.requires_grad_(False) for p in mod.parameters()]

# run_epoch uses device-aware autocast and a single GradScaler instance
def run_epoch(loader, optimizer, scaler, stage, train=True):
    model.train() if train else model.eval()
    total_loss = total_correct = total_samples = nan_count = 0
    t0 = time.time()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for feats, p_labels, g_labels, t_flags in loader:
            feats    = feats.to(device,    non_blocking=True)
            p_labels = p_labels.to(device, non_blocking=True)
            g_labels = g_labels.to(device, non_blocking=True)
            t_flags  = t_flags.to(device,  non_blocking=True)

            # Use device-aware autocast; person & group LSTMs run in float32 internally
            with torch.amp.autocast(device_type()):
                if stage == 1:
                    a_logits = model(feats, t_flags, stage=1)  # (B, T, K, C)
                    T = a_logits.shape[1]
                    p_tgt = p_labels.unsqueeze(1).expand(-1, T, -1)
                    loss = person_criterion(
                        a_logits.reshape(-1, NUM_ACTION_CLASSES),
                        p_tgt.reshape(-1))
                    mask = p_tgt.reshape(-1) != UNKNOWN_IDX
                    correct = (a_logits.reshape(-1, NUM_ACTION_CLASSES)
                               .argmax(-1)[mask] ==
                               p_tgt.reshape(-1)[mask]).sum().item() \
                              if mask.any() else 0
                    samples = mask.sum().item() if mask.any() else 1
                else:
                    g_logits, a_logits = model(feats, t_flags, stage=2)
                    T = a_logits.shape[1]
                    p_tgt = p_labels.unsqueeze(1).expand(-1, T, -1)
                    g_loss = group_criterion(g_logits, g_labels)
                    aux_loss = person_criterion(
                        a_logits.reshape(-1, NUM_ACTION_CLASSES),
                        p_tgt.reshape(-1))
                    loss = g_loss + 0.5 * aux_loss
                    correct = (g_logits.argmax(-1) == g_labels).sum().item()
                    samples = g_labels.size(0)

            if torch.isnan(loss):
                nan_count += 1
                continue

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * feats.size(0)
            total_correct += correct
            total_samples += samples

    if nan_count:
        print(f"    ⚠ {nan_count} NaN batches")
    n = len(loader.dataset)
    return (total_loss / max(n, 1),
            100.0 * total_correct / max(total_samples, 1),
            time.time() - t0)

# --- Stage 1 training (person encoder) ---
opt1 = torch.optim.Adam(get_core(model).person.parameters(),
                        lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched1 = torch.optim.lr_scheduler.StepLR(opt1, step_size=15, gamma=0.1)
scaler = torch.amp.GradScaler()
best1 = 0.

for ep in range(1, CFG["stage1_epochs"] + 1):
    tr = run_epoch(train_loader, opt1, scaler, 1, True)
    vl = run_epoch(val_loader,   opt1, scaler, 1, False)
    sched1.step()
    tag = " ◄" if vl[1] > best1 else ""
    print(f"[S1 {ep:02d}] loss={tr[0]:.4f} acc={tr[1]:.1f}% | "
          f"val={vl[0]:.4f} acc={vl[1]:.1f}% ({tr[2]:.0f}s){tag}")
    if vl[1] > best1:
        best1 = vl[1]
        torch.save(get_core(model).person.state_dict(), CFG["ckpt_dir"]/"best_stage1.pt")
print(f"✅ Stage 1 best: {best1:.1f}%")

# --- Stage 2 training (group) ---
# Load best person encoder and freeze or optionally fine-tune
ckpt_path = CFG["ckpt_dir"]/"best_stage1.pt"
if ckpt_path.exists():
    get_core(model).person.load_state_dict(torch.load(ckpt_path, map_location=device))
else:
    print("Warning: best_stage1.pt not found; continuing with current person weights.")

# Option: freeze person encoder for pure stage-2 training, or allow fine-tune
FINE_TUNE_PERSON = False  # set True to allow small-lr fine-tuning in stage 2
if not FINE_TUNE_PERSON:
    freeze(get_core(model).person)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

opt2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched2 = torch.optim.lr_scheduler.StepLR(opt2, step_size=20, gamma=0.1)
scaler = torch.amp.GradScaler()
best2 = 0.

for ep in range(1, CFG["stage2_epochs"] + 1):
    tr = run_epoch(train_loader, opt2, scaler, 2, True)
    vl = run_epoch(val_loader,   opt2, scaler, 2, False)
    sched2.step()
    tag = " ◄" if vl[1] > best2 else ""
    print(f"[S2 {ep:02d}] loss={tr[0]:.4f} acc={tr[1]:.1f}% | "
          f"val={vl[0]:.4f} acc={vl[1]:.1f}% ({tr[2]+vl[2]:.0f}s){tag}")
    if vl[1] > best2:
        best2 = vl[1]
        torch.save({"epoch": ep, "val_acc": vl[1],
                    "model_state": get_core(model).state_dict(),
                    "group_vocab": GROUP_VOCAB, "action_vocab": ACTION_VOCAB},
                   CFG["ckpt_dir"]/"best_full_model.pt")

print(f"\n✅ Done!  Best val acc: {best2:.1f}%  (paper: 51.1%)")

# Safe per-class breakdown (only if checkpoint exists)
best_ckpt = CFG["ckpt_dir"]/"best_full_model.pt"
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=device)
    get_core(model).load_state_dict(ckpt["model_state"])
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for feats, _, g_labels, t_flags in val_loader:
            g_out, _ = model(feats.to(device), t_flags.to(device), stage=2)
            preds.append(g_out.argmax(-1).cpu())
            labels.append(g_labels)
    preds  = torch.cat(preds)
    labels = torch.cat(labels)

    print(f"\n{'Class':12s}  {'Acc':>7}  {'N':>5}")
    print("-"*28)
    for i in range(NUM_GROUP_CLASSES):
        m = labels == i
        if not m.any(): continue
        c = (preds[m] == i).sum().item()
        print(f"{INV_VOCAB[i]:12s}  {100*c/m.sum():>6.1f}%  {m.sum():>5}")
    print(f"\nOverall: {100*(preds==labels).float().mean():.1f}%")
else:
    print("No best_full_model.pt found; skipping per-class breakdown.")


Loading dataset...
  Valid            : 3237
  Standing kept    : 313/2091
  Final dataset    : 1459
  Train:1168  Val:291  Batches:36/5

DataParallel: 2 GPUs


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


[S1 01] loss=2.2858 acc=21.9% | val=2.1573 acc=18.2% (1s) ◄
[S1 02] loss=2.0562 acc=22.6% | val=2.0429 acc=19.0% (1s) ◄
[S1 03] loss=2.0191 acc=22.0% | val=1.9983 acc=21.1% (1s) ◄
[S1 04] loss=1.9909 acc=23.1% | val=2.0021 acc=22.2% (1s) ◄
[S1 05] loss=1.9581 acc=23.8% | val=2.0119 acc=27.2% (1s) ◄
[S1 06] loss=1.9460 acc=22.9% | val=1.9717 acc=19.9% (1s)
[S1 07] loss=1.8982 acc=23.2% | val=1.9800 acc=32.8% (1s) ◄
[S1 08] loss=1.8916 acc=25.1% | val=1.9759 acc=23.8% (1s)
[S1 09] loss=1.8642 acc=24.4% | val=1.9935 acc=27.7% (1s)
[S1 10] loss=1.8674 acc=25.0% | val=2.0977 acc=24.0% (1s)
[S1 11] loss=1.8510 acc=24.4% | val=2.0326 acc=20.0% (1s)
[S1 12] loss=1.8286 acc=24.7% | val=1.9723 acc=27.4% (1s)
[S1 13] loss=1.8113 acc=25.7% | val=2.0660 acc=22.6% (1s)
[S1 14] loss=1.7873 acc=26.0% | val=1.9776 acc=21.2% (1s)
[S1 15] loss=1.7551 acc=26.6% | val=2.0317 acc=28.2% (1s)
[S1 16] loss=1.7178 acc=27.8% | val=1.9641 acc=23.7% (1s)
[S1 17] loss=1.7033 acc=27.2% | val=1.9910 acc=25.7% (1s)
[S